In [6]:
%load_ext autoreload
%autoreload 2
import os,shutil
import numpy as np
import pandas as pd
import pickle as pkl

from inDecay import PATH
os.chdir(PATH.main_dir)

from qrguide import analysis_fn

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
from inDecay import my_utils

In [23]:
BOB = 'ST_June_2017_BOB_LV7A_DPI7'
ETG = 'ST_June_2017_E14TG2A_LV7A_DPI7'

celltype_rename = { BOB:'iPSC', 
                    ETG:'mESC'}

In [ ]:
def read_pkl(path):
    with open(path, 'rb') as f:
        Y = pkl.load(f)
    f.close()
    return Y

def evalute_fn(Y_true_path, Y_pred_path):
    Y_pred =read_pkl(Y_pred_path)
    Y = read_pkl(Y_true_path)

    eval_json = analysis_fn.assessment_recipe_forecast(Y, Y_pred)
    eval_json.update(analysis_fn.assessment_recipe_IDL_forecast(Y, Y_pred))

    eval_df = pd.json_normalize(eval_json)
    return eval_df


# find the ckpt

In [50]:
pred_pkl_path_dict = {}
true_pkl_path_dict = {}


for feat_i in range(0,300,30):

    feat_range = f"{feat_i}:{feat_i+30}" 
    pred_pkl_path_dict[feat_range] = {}
    true_pkl_path_dict[feat_range] = {}

    for exp in [BOB, ETG]:
        
        celltype = celltype_rename[exp]

        # point to lightning saving path    
        log_path = os.path.join(
            f"pl_trainer_log/ST_featv5_{feat_range}_ST_Decay_identity",
            exp,
            "lightning_logs"
        )

        if not os.path.exists(log_path):
            continue

        versions = [f for f in os.listdir(log_path) if f.startswith('version')]

        # remove empty versions
        if len(versions) > 1:
            for v in versions:
                 if 'checkpoints' not in os.listdir(os.path.join(log_path, v)):
                     shutil.rmtree( os.path.join(log_path, v))
                     
        try:
            pkl_path = my_utils.find_ckpt(log_path).replace(".ckpt", "TestPred.pkl")
            pred_pkl_path_dict[feat_range][celltype] = pkl_path

            Ytrue_path = log_path.replace("lightning_logs","ForeCast_TestY.pkl")
            true_pkl_path_dict[feat_range][celltype] = Ytrue_path

        except FileNotFoundError:
            # skip some feat_i or celltype if no valid ckpt found
            continue    

    print(f"for {feat_range}, checkpoints of {len(pred_pkl_path_dict[feat_range])} cells were found")

for 0:30, checkpoints of 2 cells were found
for 30:60, checkpoints of 2 cells were found
for 60:90, checkpoints of 2 cells were found
for 90:120, checkpoints of 2 cells were found
for 120:150, checkpoints of 2 cells were found
for 150:180, checkpoints of 2 cells were found
for 180:210, checkpoints of 2 cells were found
for 210:240, checkpoints of 2 cells were found
for 240:270, checkpoints of 2 cells were found
for 270:300, checkpoints of 2 cells were found


# evaluation

In [54]:
perform = []
for feat_range, pred_pkls in pred_pkl_path_dict.items():

    true_pkls = true_pkl_path_dict[feat_range]
    
    for cell in celltype_rename.values():
        df = evalute_fn(true_pkls[cell], pred_pkls[cell])
        df['celltype'] = cell
        df['feat_range'] = feat_range

        perform.append(df)


featv5_df = pd.concat(perform, axis=0).reset_index().iloc[:, 1:]

In [55]:
featv5_df.query("`celltype` == 'iPSC'")

,KL divergence,Top5 events recall,Top10 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top5_IDL,Top10_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,celltype,feat_range
0,10.056778,0.327449,2.103266,0.267639,1.428067,2.720212,1.958524,0.276258,2.072374,0.274957,0.059392,0.217196,iPSC,0:30
2,11.404818,0.060900,0.345102,0.079707,0.356575,1.150927,3.012145,0.098853,0.658429,0.286802,0.017390,0.124447,iPSC,30:60
4,11.141659,0.011474,0.038835,0.025869,0.120918,0.414828,2.820195,0.022065,0.561342,0.284249,0.000206,0.027356,iPSC,60:90
6,10.448299,0.538394,0.980583,0.032125,1.450132,2.259488,2.363247,0.033539,0.584289,0.280006,0.025242,0.033139,iPSC,90:120
8,11.313068,0.000000,0.012357,0.015816,0.002648,0.025596,3.109140,0.016770,0.542807,0.285982,0.014243,0.021607,iPSC,120:150
10,11.900195,0.000000,0.001765,0.005838,0.000000,0.002648,3.860220,0.019417,0.569285,0.288719,0.002202,0.023699,iPSC,150:180
12,11.746377,0.142983,0.197705,0.005255,0.142983,0.200353,3.587947,0.038835,0.643425,0.286190,0.051577,0.032713,iPSC,180:210
14,12.147201,0.000000,0.003530,0.014710,0.000000,0.004413,4.096391,0.010591,0.567520,0.289108,0.032132,0.021694,iPSC,210:240
16,10.258394,0.439541,1.640777,0.158517,1.325684,2.497793,2.159178,0.037070,0.973522,0.278558,0.054434,0.075641,iPSC,240:270
18,12.192693,0.024713,0.062665,0.068297,0.025596,0.068844,3.971335,0.061783,0.642542,0.290325,0.039707,0.076296,iPSC,270:300


In [56]:
featv5_df.query("`celltype` == 'mESC'")

,KL divergence,Top5 events recall,Top10 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top5_IDL,Top10_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,celltype,feat_range
1,12.589545,0.000000,0.000883,0.003376,0.000000,0.000883,5.559279,0.016770,0.401589,0.260314,1.018913e-02,-0.019862,mESC,0:30
3,10.497226,0.017652,0.044131,0.117771,0.431598,1.058252,2.714250,0.069726,0.346867,0.254149,8.700070e-02,-0.014695,mESC,30:60
5,7.614685,1.567520,3.950574,0.233114,1.769638,4.385702,1.582771,0.483672,4.699029,0.238343,1.329089e-02,0.516663,mESC,60:90
7,11.417185,0.011474,0.032657,0.063203,0.011474,0.032657,3.483491,0.050309,0.388350,0.257159,6.346772e-02,-0.017030,mESC,90:120
9,9.951731,0.063548,0.388350,0.147419,0.972639,1.485437,2.377663,0.012357,0.223301,0.251458,2.294092e-03,-0.041050,mESC,120:150
11,9.396520,0.375993,1.589585,0.117081,0.947926,2.511915,2.067796,0.209179,1.522507,0.248059,3.137269e-02,0.257812,mESC,150:180
13,11.585173,0.002648,0.003530,0.011774,0.002648,0.003530,4.204984,0.012357,0.346867,0.258744,2.342687e-07,-0.023026,mESC,180:210
15,11.657201,0.015004,0.037952,0.064862,0.015004,0.038835,3.701280,0.068844,0.401589,0.259229,1.332920e-02,0.001881,mESC,210:240
17,11.518294,0.000000,0.002648,0.028381,0.000000,0.003530,3.603479,0.014122,0.346867,0.257579,3.383553e-02,-0.026672,mESC,240:270
19,7.523951,0.987643,3.370697,0.140685,1.465137,3.915269,1.525576,0.750221,5.353045,0.236603,1.022052e-01,0.570477,mESC,270:300


# read feature names

In [59]:
from inDecay import alignmap 

readFeaturesData = alignmap.fft.readFeaturesData

In [38]:
filename = "Oligo50293_GGATCGAGGCTATAAATCGC_features.txt"
feature_path = os.path.join(PATH.data_dir, 'IndelFeature_result', filename)

In [39]:
feature_df, cols = alignmap.fft.readFeaturesData(feature_path)
sinlge_col = [col for col in feature_df.columns if not col.startswith('PW_')]

feat_df = feature_df[sinlge_col]

In [42]:
feat_names = list(feat_df.columns)

In [44]:
for i in range(0,len(feat_names), 10):
    print(feat_names[i:i+10])

['Any Insertion', 'I1', 'I2', 'Any Deletion', 'D1', 'D2-3', 'D4-7', 'D8-12', 'D>12', 'DL-1--1']
['DL-2--2', 'DL-3--3', 'DL-4--6', 'DL-7--10', 'DL-11--15', 'DL-16--30', 'DL<-30', 'DL>=0', 'DR0-0', 'DR1-1']
['DR2-2', 'DR3-5', 'DR6-9', 'DR10-14', 'DR15-29', 'DR<0', 'DR=>30', 'IL-1--1', 'IL-2--2', 'IL-3--3']
['IL<-3', 'IL>=0', 'I1Rpt', 'I1NonRpt', 'I2Rpt', 'I2NonRpt', 'I1_A', 'I2_AA', 'I2_AT', 'I2_AG']
['I2_AC', 'I1_T', 'I2_TA', 'I2_TT', 'I2_TG', 'I2_TC', 'I1_G', 'I2_GA', 'I2_GT', 'I2_GG']
['I2_GC', 'I1_C', 'I2_CA', 'I2_CT', 'I2_CG', 'I2_CC', 'CS-5_NT=A', 'CS-5_NT=T', 'CS-5_NT=G', 'CS-5_NT=C']
['CS-4_NT=A', 'CS-4_NT=T', 'CS-4_NT=G', 'CS-4_NT=C', 'CS-3_NT=A', 'CS-3_NT=T', 'CS-3_NT=G', 'CS-3_NT=C', 'CS-2_NT=A', 'CS-2_NT=T']
['CS-2_NT=G', 'CS-2_NT=C', 'CS-1_NT=A', 'CS-1_NT=T', 'CS-1_NT=G', 'CS-1_NT=C', 'CS0_NT=A', 'CS0_NT=T', 'CS0_NT=G', 'CS0_NT=C']
['CS1_NT=A', 'CS1_NT=T', 'CS1_NT=G', 'CS1_NT=C', 'CS2_NT=A', 'CS2_NT=T', 'CS2_NT=G', 'CS2_NT=C', 'CS3_NT=A', 'CS3_NT=T']
['CS3_NT=G', 'CS3_NT=C',

In [49]:
for i in range(50, 120, 10):
    print(feat_names[i:i+10])

['I2_GC', 'I1_C', 'I2_CA', 'I2_CT', 'I2_CG', 'I2_CC', 'CS-5_NT=A', 'CS-5_NT=T', 'CS-5_NT=G', 'CS-5_NT=C']
['CS-4_NT=A', 'CS-4_NT=T', 'CS-4_NT=G', 'CS-4_NT=C', 'CS-3_NT=A', 'CS-3_NT=T', 'CS-3_NT=G', 'CS-3_NT=C', 'CS-2_NT=A', 'CS-2_NT=T']
['CS-2_NT=G', 'CS-2_NT=C', 'CS-1_NT=A', 'CS-1_NT=T', 'CS-1_NT=G', 'CS-1_NT=C', 'CS0_NT=A', 'CS0_NT=T', 'CS0_NT=G', 'CS0_NT=C']
['CS1_NT=A', 'CS1_NT=T', 'CS1_NT=G', 'CS1_NT=C', 'CS2_NT=A', 'CS2_NT=T', 'CS2_NT=G', 'CS2_NT=C', 'CS3_NT=A', 'CS3_NT=T']
['CS3_NT=G', 'CS3_NT=C', 'M_CS-2_-3_NT=A', 'M_CS-2_-3_NT=T', 'M_CS-2_-3_NT=G', 'M_CS-2_-3_NT=C', 'M_CS-1_-3_NT=A', 'M_CS-1_-3_NT=T', 'M_CS-1_-3_NT=G', 'M_CS-1_-3_NT=C']
['M_CS-1_-2_NT=A', 'M_CS-1_-2_NT=T', 'M_CS-1_-2_NT=G', 'M_CS-1_-2_NT=C', 'M_CS0_-3_NT=A', 'M_CS0_-3_NT=T', 'M_CS0_-3_NT=G', 'M_CS0_-3_NT=C', 'M_CS0_-2_NT=A', 'M_CS0_-2_NT=T']
['M_CS0_-2_NT=G', 'M_CS0_-2_NT=C', 'M_CS0_-1_NT=A', 'M_CS0_-1_NT=T', 'M_CS0_-1_NT=G', 'M_CS0_-1_NT=C', 'M_CS1_-3_NT=A', 'M_CS1_-3_NT=T', 'M_CS1_-3_NT=G', 'M_CS1_-3_NT=C']


# read_data

In [60]:
from scripts.STfeatv5_inDecay import read_data

In [57]:
from inDecay import reader
from inDecay import ForeCast_features as fft

In [61]:
reference_path = os.path.join(PATH.data_dir, "SelfTarget_NewScaffold.fasta")
ref_lookup = reader.get_reference()

In [66]:
test_Oligo = "Oligo50293"
Guide, refseq, pamsite, Strand = ref_lookup[test_Oligo]
idfgen_file = my_utils.get_indelgen_file(test_Oligo, Guide)
idfgen = pd.read_table(idfgen_file, skiprows=1, names=['Identifier', 'n_coevent', 'loc'])

In [67]:
idfgen

,Identifier,n_coevent,loc
0,D10_L-10C2R3,3,"[(43,54,),(44,55,),(45,56,)]"
1,D10_L-11R0,1,"[(42,53,)]"
2,D10_L-1R10,1,"[(52,63,)]"
3,D10_L-2R9,1,"[(51,62,)]"
4,D10_L-3R8,1,"[(50,61,)]"
...,...,...,...
374,I2_L-1C4R4,1,"[(52,53,CG)]"
375,I2_L-1R0,9,"[(52,53,AA),(52,53,TA),(52,53,GA),(52,53,AG),(..."
376,I2_L-2C1R0,2,"[(52,53,TT),(52,53,GT)]"
377,I2_L-2C2R1,1,"[(52,53,CT)]"


In [71]:
import Bio

In [117]:
A,T,G,C = 'A','T','G','C'
AA,AT,AC,AG,CG,CT,CA,CC = 'AA','AT','AC','AG','CG','CT','CA','CC'
GT,GA,GG,GC,TA,TG,TC,TT = 'GT','GA','GG','GC','TA','TG','TC','TT'

is_reverse = (Strand == "REVERSE")

cut_site = pamsite - 3

feature_matrix = []
indels = []
for i, row in idfgen.iterrows():
    indel = row['Identifier']
    indel_locs = eval(row['loc'])
    for indel_loc in indel_locs:
        ins_seq = indel_loc[2] if len(indel_loc) > 2 else ''
        left = indel_loc[0] if not is_reverse else (78 - indel_loc[1]) 
        right = indel_loc[1] if not is_reverse else (78 - indel_loc[0]) 
        ins_seq = ins_seq if not is_reverse else Bio.Seq.reverse_complement(ins_seq)
        indel_details = (refseq, cut_site, left, right, ins_seq)

        lin_fts, feat_name = fft.features_SeqMatches(indel_details)

        feature_matrix.append(lin_fts)
        indels.append(indel)

feature_df = pd.DataFrame(feature_matrix, columns=feat_name)
feature_df['Identifier'] = indels

In [120]:
feature_df.groupby('Identifier').any()

,M_L-3_R-3,X_L-3_R-3,M_L-3_R-2,X_L-3_R-2,M_L-3_R-1,X_L-3_R-1,M_L-3_R0,X_L-3_R0,M_L-3_R1,X_L-3_R1,...,M_L2_R-2,X_L2_R-2,M_L2_R-1,X_L2_R-1,M_L2_R0,X_L2_R0,M_L2_R1,X_L2_R1,M_L2_R2,X_L2_R2
Identifier,,,,,,,,,,,,,,,,,,,,,
D10_L-10C2R3,False,True,True,True,False,True,True,True,True,True,...,True,True,False,True,False,True,False,True,False,True
D10_L-11R0,False,True,False,True,False,True,True,False,False,True,...,False,True,False,True,True,False,False,True,True,False
D10_L-1R10,False,True,False,True,False,True,False,True,True,False,...,True,False,False,True,False,True,False,True,False,True
D10_L-2R9,False,True,False,True,False,True,False,True,False,True,...,False,True,False,True,True,False,True,False,False,True
D10_L-3R8,False,True,True,False,False,True,False,True,False,True,...,False,True,True,False,True,False,False,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
I2_L-1C4R4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
I2_L-1R0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
I2_L-2C1R0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [116]:
feature_matrix = []
for x in cal_feat_input:
    all_features = []
    lin_fts, feat_name = fft.features_SeqMatches(x)
    # for fname in lin_fts:
    #     all_features.extend(lin_fts[fname][0])

    feature_matrix.append(lin_fts)
    
feature_matrix = np.stack(feature_matrix)
feature_df = pd.DataFrame(feature_matrix, columns=feat_name)

In [102]:
feature_matrix.shape

(500, 40)

In [108]:
func_2_feat = {k:list(v)[1] for k,v in lin_fts[0].items()}

AttributeError: 'bool' object has no attribute 'items'

In [114]:
feature_list = ["feature_InsSize", "feature_DelSize", "feature_DelLoc", "feature_InsLoc", "feature_I1or2Rpt","feature_InsSeq", "feature_LocalCutSiteSequence", "feature_LocalCutSiteSeqMatches", "feature_LocalRelativeSequence", "features_SeqMatches", "feature_microhomology"]

cumsum_i = 0
for feat in feature_list:
    len_feat = len(func_2_feat[feat][1])
    print(feat, cumsum_i, ":", cumsum_i + len_feat)
    cumsum_i += len_feat

feature_InsSize 0 : 3
feature_DelSize 3 : 9
feature_DelLoc 9 : 27
feature_InsLoc 27 : 32
feature_I1or2Rpt 32 : 36
feature_InsSeq 36 : 56
feature_LocalCutSiteSequence 56 : 92
feature_LocalCutSiteSeqMatches 92 : 132
feature_LocalRelativeSequence 132 : 180
features_SeqMatches 180 : 252
feature_microhomology 252 : 273


In [115]:
92 - 56

36

In [89]:
list(lin_fts[0].values())[0][0]

([False, False, False], ['Any Insertion', 'I1', 'I2'])

In [84]:
feature_X = [list(fts_dict.values())[0] for fts_dict in lin_fts]

In [85]:
feature_X

[([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any 